# HEST-1k Data Exploration

This notebook looks at the HEST-1k sample catalog (`HEST_v1_1_0.csv`) to help us choose which data to use.

It counts, for human samples only:
1. How many samples each organ has
2. How many samples each organ has per technology (ST, Visium, Visium HD, Xenium)
3. How many patients and studies each organ + technology group has

This tells us how many patients we can use for a patient-level train/test split.
Note: a patient count of 0 means the patient info is missing, not zero.

In [2]:
%pip install -q huggingface_hub pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
from huggingface_hub import login
login()

In [4]:
from huggingface_hub import hf_hub_download
import pandas as pd

meta_path = hf_hub_download(
    repo_id="MahmoodLab/hest",
    repo_type="dataset",
    filename="HEST_v1_1_0.csv",
)

meta = pd.read_csv(meta_path)

print("Total samples:", len(meta))

Total samples: 1229


In [5]:
organ_counts = meta[meta["species"] == "Homo sapiens"]["organ"].value_counts()

print("Number of different organs:", len(organ_counts))
print()
print(organ_counts.to_string())

Number of different organs: 18

organ
Breast        125
Bowel          84
Skin           80
Kidney         64
Heart          62
Prostate       60
Lung           59
Brain          43
Uterus         17
Liver          16
Pancreas        8
Bladder         6
Eye             6
Lymphoid        5
Cervix          5
Lymph node      5
Ovary           3
Bone            1


In [6]:
human = meta[meta["species"] == "Homo sapiens"]

tech_table = pd.crosstab(human["organ"], human["st_technology"])

tech_table["Total"] = tech_table.sum(axis=1)
tech_table = tech_table.sort_values("Total", ascending=False)

print(tech_table.to_string())

st_technology  Spatial Transcriptomics  Visium  Visium HD  Xenium  Total
organ                                                                   
Breast                             108       8          0       9    125
Bowel                                0      73          5       6     84
Skin                                12      63          0       5     80
Kidney                               0      62          0       2     64
Heart                               19      42          0       1     62
Prostate                            24      35          0       1     60
Lung                                 0      37          0      22     59
Brain                                0      42          0       1     43
Uterus                               0      17          0       0     17
Liver                                0      14          0       2     16
Pancreas                             0       4          1       3      8
Eye                                  0       6     

In [7]:
print(meta.columns.tolist())

['dataset_title', 'id', 'image_filename', 'organ', 'disease_state', 'oncotree_code', 'species', 'patient', 'st_technology', 'data_publication_date', 'license', 'study_link', 'download_page_link1', 'inter_spot_dist', 'spot_diameter', 'spots_under_tissue', 'preservation_method', 'nb_genes', 'treatment_comment', 'pixel_size_um_embedded', 'pixel_size_um_estimated', 'magnification', 'fullres_px_width', 'fullres_px_height', 'tissue', 'disease_comment', 'subseries', 'hest_version_added']


In [8]:
human = meta[meta["species"] == "Homo sapiens"]

patient_table = human.groupby(["organ", "st_technology"]).agg(
    samples=("id", "count"),
    patients=("patient", "nunique"),
    studies=("dataset_title", "nunique"),
)

print(patient_table.to_string())

                                    samples  patients  studies
organ      st_technology                                      
Bladder    Visium                         6         5        2
Bone       Xenium                         1         0        1
Bowel      Visium                        73         8       10
           Visium HD                      5         1        2
           Xenium                         6         1        3
Brain      Visium                        42        12        9
           Xenium                         1         0        1
Breast     Spatial Transcriptomics      108        31        3
           Visium                         8         0        8
           Xenium                         9         5        4
Cervix     Visium                         5         0        2
Eye        Visium                         6         0        2
Heart      Spatial Transcriptomics       19         0        1
           Visium                        42         1  

In [9]:
missing = human["patient"].isna().sum()
print("Samples with no patient info:", missing)

Samples with no patient info: 255
